# Explore the Yelp data (McDonald's and IKEA)

Before a model reads any review, we look at the data: which locations are reviewed, how the ratings are distributed, and who writes reviews.

In this notebook, we will:
1. Import the McDonald's and IKEA locations and compare their ratings.
2. Import all IKEA reviews and explore stars, time, stores and review length.
3. Read some reviews and look for what a single star rating cannot show.

**Time:** about 15 minutes

**Before you start: get the data from Moodle**
1. Download `llm_fundamentals_data.zip` from Moodle (Week 10).
2. Drag the zip file into the file Explorer, onto the folder `data`.
3. In the terminal, run: `python -m zipfile -e data/llm_fundamentals_data.zip data`

After unzipping, the folder `data` looks like this:

```
data/
├── businesses_mcdonalds_ikea.csv
├── ikea_reviews.csv
├── fewshot_examples.json
├── TERMS_OF_USE.txt
├── llm_fundamentals_data.zip
├── README.md
├── processed/
└── raw/
```

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import pandas as pd
import matplotlib.pyplot as plt

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Import the business records

In [ ]:
# Read the McDonald's and IKEA locations (data from Moodle, see README: Data)
df_bus = pd.read_csv(
    '../data/businesses_mcdonalds_ikea.csv', sep=',', encoding='utf-8'
)

# Shape and first rows
print(df_bus.shape)
df_bus.head(5)

## Number of locations and reviews per chain

In [ ]:
# Locations, reviews and mean rating per chain
summary = df_bus.groupby('chain').agg(
    locations=('business_id', 'count'),
    reviews=('review_count', 'sum'),
    mean_stars=('stars', 'mean'),
)
summary.round(2)

## Average rating: per location vs. per review

Weighting by the number of reviews treats every review equally.

In [ ]:
# Average of the location ratings, weighted by the number of reviews
df_bus['stars_x_reviews'] = df_bus['stars'] * df_bus['review_count']
grouped = df_bus.groupby('chain')
summary['weighted_stars'] = (
    grouped['stars_x_reviews'].sum() / grouped['review_count'].sum()
)

summary.round(2)

## Distribution of the location ratings

In [ ]:
# One bar chart per chain
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (chain, group) in zip(axes, df_bus.groupby('chain')):
    group['stars'].value_counts().sort_index().plot.bar(ax=ax, color='steelblue')
    ax.set_title(f'{chain}: {len(group)} locations')
    ax.set_xlabel('Stars of the location (rounded to 0.5)')
    ax.set_ylabel('Number of locations')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## Locations per state

In [ ]:
# Number of locations per state
pd.crosstab(df_bus['state'], df_bus['chain']).sort_values(
    "McDonald's", ascending=False
)

## The IKEA locations

In [ ]:
# All IKEA records
df_bus.loc[
    df_bus['chain'] == 'IKEA', ['name', 'city', 'state', 'stars', 'review_count']
]

## Import the IKEA reviews

In [ ]:
# Read all IKEA reviews (data from Moodle, see README: Data)
df = pd.read_csv(
    '../data/ikea_reviews.csv', sep=',', encoding='utf-8', parse_dates=['date']
)

# Shape and first rows
print(df.shape)
df.head(5)

## Star distribution of the reviews

In [ ]:
# Number of reviews per star rating
counts = df['stars'].value_counts().sort_index()

ax = counts.plot.bar(color='steelblue', figsize=(6, 4))
ax.bar_label(ax.containers[0])
plt.title(f'All {len(df):,} IKEA reviews by rating')
plt.xlabel('Stars')
plt.ylabel('Number of reviews')
plt.xticks(rotation=0)
plt.show()

Many 4s and 5s, many 1s, few in between: the typical **J-shape** of online reviews.

## Reviews over time

In [ ]:
# Number of reviews and mean rating per year
per_year = df.groupby(df['date'].dt.year).agg(
    reviews=('review_id', 'count'),
    mean_stars=('stars', 'mean'),
)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(per_year.index, per_year['reviews'], color='lightsteelblue')
ax1.set_xlabel('Year')
ax1.set_ylabel('Number of reviews')

ax2 = ax1.twinx()
ax2.plot(per_year.index, per_year['mean_stars'], color='darkred', marker='o')
ax2.set_ylabel('Mean stars', color='darkred')
ax2.set_ylim(1, 5)

plt.title('IKEA reviews per year')
plt.show()

## Ratings by store

In [ ]:
# Reviews, mean rating and time span per store
df.groupby(['store', 'city']).agg(
    reviews=('review_id', 'count'),
    mean_stars=('stars', 'mean'),
    first_review=('date', 'min'),
    last_review=('date', 'max'),
).sort_values('reviews', ascending=False).round(2)

## Review length

In [ ]:
# Number of words per review
df['words'] = df['text'].str.split().str.len()
print(df['words'].describe().round(1))

# Review length by star rating
df.boxplot(column='words', by='stars', showfliers=False, figsize=(6, 4))
plt.suptitle('')
plt.title('Review length by stars (without outliers)')
plt.xlabel('Stars')
plt.ylabel('Words')
plt.show()

## Read some reviews

In [ ]:
# One random review per star rating
for stars, group in df.groupby('stars'):
    row = group.sample(1, random_state=42).iloc[0]
    print(f"{stars} stars | {row['store']}, {row['city']} | {row['date']:%d %b %Y}")
    print(row['text'][:500], '\n')

## Search for a topic

In [ ]:
# TODO: try other topics, e.g. 'assembly', 'return', 'parking', 'staff'
topic = 'meatball'

hits = df[df['text'].str.contains(topic, case=False)]
print(f"{len(hits)} reviews mention '{topic}'")
print(f"Mean stars: {hits['stars'].mean():.2f} (all reviews: {df['stars'].mean():.2f})")
hits[['stars', 'store', 'text']].head(5)

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')